# Quantitative Alpha Research Walkthrough
## Predicting Daily Cross-Sectional Return Rankings (US Equity - Russell 1000)

This notebook demonstrates the end-to-end quantitative workflow:
1. **Universe Setup**: Loading the Russell 1000 large-cap constituent universe (>= 500 stocks).
2. **Data Pipeline**: In-sample data loading (< 2024-01-01) with leakage prevention.
3. **Alpha Generation**: 5-Day Reversal benchmark and WorldQuant Brain-style operators.
4. **Long-Short Neutralization**: Dollar-neutral positions ($\sum \alpha_i = 0$, $L_1$ leverage scaling).
5. **Backtesting & Evaluation**: PnL, Annualized Returns, Sharpe Ratio, and Rank IC.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.universe import build_or_load_universe
from src.data.split import load_in_sample_data
from src.alphas.reversal_5d import Reversal5D
from src.alphas.alpha_library import get_all_alphas
from src.portfolio.combiner import MetaAlphaCombiner
from src.backtest.engine import BacktestEngine

print('All modules loaded successfully!')

### Step 1: Universe Inspection

In [ ]:
tickers = build_or_load_universe(min_stocks=500)
print(f'Total Universe Size: {len(tickers)} stocks')
print('First 10 constituents:', tickers[:10])

### Step 2: Load In-Sample Data (< 2024-01-01)

In [ ]:
# Load strictly in-sample cross-sectional matrices
try:
    data = load_in_sample_data()
    print(f'In-Sample Date Range: {data.get_dates()[0]} to {data.get_dates()[-1]}')
    print(f'Number of stocks: {len(data.get_tickers())}')
except Exception as e:
    print(f'Note: Run python scripts/02_download_data.py to populate historical data. Error: {e}')

### Step 3: Run Benchmark 5-Day Reversal Strategy

In [ ]:
strategy = Reversal5D(lookback=5)
engine = BacktestEngine(transaction_cost_bps=5.0, target_leverage=1.0)

# report = engine.run(strategy, data)
# print(f"Net Sharpe: {report['net_sharpe_ratio']:.2f}")
# print(f"Net Annualized Return: {report['net_annualized_return']:.2%}")